In [ ]:
import re
import pandas as pd
import copy
import numpy as np
import evaluate
from docx import Document
import os
import unittest
from datasets import load_dataset
import pickle
import matplotlib.pyplot as plt
from bleurt import score
class DataLoader:
    """
    Loader for benchmarking datasets to ensure universal formatting. To be used in conjunction with DyslexiaInjector.
    ...
    Attributes
    ----------
    path: str
        Path to csv, txt or docx file of the data. In the case of CSV there should only be 1 column
    data: list
        A list of striings
    dataset_name: str
        Name of the dataset that is used when saving the data
    ...
    Methods
    -------
    parse_txt(path)
        Parses a txt file and returns a list of strings
    fix_format(sentence)
        Fixes the formatting of a sentence
    save_as_txt(path)
        Saves the data as a txt file
    save_as_csv(path)
        Saves the data as a csv file
    save_as_docx(path)
        Saves the data as a docx file
    get_data()
        Returns the data
    create_deepcopy()
        Returns a deepcopy of the DataLoader instance
    get_name()
        Returns the dataset name
    get_number_of_sentences()
        Returns the number of sentences in the data
    get_number_of_words()
        Returns the number of words in the data
    get_number_of_letters()
        Returns the number of letters in the data
    edit_distance(reference_sentence, sentence)
        Returns the number of edits required to transform reference_sentence into sentence at word level
        edits include insertions, deletions and substitutions
        based on levenshtein distance
        also returns a dictionary of substitutions, insertions and deletions
    get_edit_distance(reference, manual_wer=False)
        Returns the number of edits required to transform data into reference at word level, substitutions, insertions and deletions the associated dictionaries
        and the WER (withouth alignment) if manual_wer is set to True
    get_individual_edit_distance(reference)
        Returns the number of edits required to transform data into reference at word level for each individual sentence
    combine_nested_dict(dict1, dict2)
        Combines two nested dictionaries
    combine_dicts(dict1, dict2)
        Combines two dictionaries
    get_bleue_score(reference)
        Returns bleu score of the data against a reference
    get_wer(reference)
        Returns the Word Error Rate (WER) of the data against a reference. With word alignment
    get_bert_score(reference)
        Returns the BERT Score similarity score of the data against a reference
    get_LaBSE(reference, model=None, tokenizer=None)
        Returns the LaBSE similarity score of the data against a reference which is a l2 norm between the reference and target sentences score.
        Score of 1 means the sentences are identical, closer to 0 means they are less similar semantically.
    ...

    Usage
    -------
    >>> from datasets import load_dataset
    >>> from DataLoader import DataLoader
    >>> dataset_wmt_enfr = load_dataset("wmt14",'fr-en', split='test')
    >>> to_translate = []
    >>> for i in range(len(dataset_wmt_enfr)):
    >>>     to_translate.append(dataset_wmt_enfr[i]['translation']['en'])
    >>> loader = DataLoader(data=to_translate, dataset_name="wmt14_enfr")
    >>> loader.save_as_txt("wmt14_enfr.txt")
    We can also use the text file to create a new DataLoader instance
    >>> loader2 = DataLoader(path="wmt14_enfr.txt", dataset_name="wmt14_enfr")
    """
    # Constructor
    def __init__(self, path=None, data=None, dataset_name=""):
        self.dataset_name = dataset_name
        if data is None and path is not None:
            #check path to see if file is txt or csv
            file_type = path.split(".")[-1]
            if file_type == "txt":
                self.data = self.parse_txt(path)
                self.data = [self.fix_format(sentence) for sentence in self.data]
            elif file_type == "csv":
                self.data = pd.read_csv(path, header=None)
                self.data = self.data[0].tolist()
                #fix any formatting issues
                self.data = [self.fix_format(sentence) for sentence in self.data]
            elif file_type == "docx":
                doc = Document(path)
                self.data = [self.fix_format(paragraph.text) for paragraph in doc.paragraphs]
            else:
                raise Exception("Invalid file type")
        elif data is not None:
            #check if data is a list or a df
            if isinstance(data, list):
                #format each sentence in data
                self.data = [self.fix_format(sentence) for sentence in data]
            else:
                raise Exception("Invalid data type, please pass in a list of sentences")
        else:
            raise Exception("Please pass in a path or data")

    def parse_txt(self, path):
        output = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                output.append(self.fix_format(line))
        return output
                
    def fix_format(self, sentence):
        #remove spacing before punctuation
        sentence = re.sub(r'\s([?.!,"](?:\s|$))', r'\1', sentence)
        #replace any double spaces with single space
        sentence = re.sub(r'\s+', ' ', sentence)
        #remove any leading or trailing spaces
        sentence = sentence.strip()
        #make all quotes (german and french) english double quotes
        sentence = re.sub(r'«|»|„|“', '"', sentence)
        #make all single quotes english single quotes
        sentence = re.sub(r'‘|’', "'", sentence)
        #make all french guillemets english double quotes
        sentence = re.sub(r'‹|›', '"', sentence)
        #if sentence begins and ends with quotes and there are only two, remove them
        if sentence[0] == '"' and sentence[-1] == '"' and sentence.count('"') == 2:
            sentence = sentence[1:-1]
        elif sentence[0] == "'" and sentence[-1] == "'" and sentence.count("'") == 2:
            sentence = sentence[1:-1]
        return sentence

    def save_as_txt(self, path):
        with open(path, "w", encoding="utf-8") as f:
            for sentence in self.data:
                f.write(f"{sentence}\n")
        print(f"Saved {self.dataset_name} to {path}")
        return
    
    def save_as_csv(self, path):
        df = pd.DataFrame(self.data)
        df.to_csv(path, index=False, header=False, encoding='utf-8')
        print(f"Saved {self.dataset_name} to {path}")
        return
    
    def save_as_docx(self, path):
        document = Document()
        for sentence in self.data:
            document.add_paragraph(sentence)
        document.save(path)
        print(f"Saved {self.dataset_name} to {path}")
        return

    def get_data(self):
        return self.data

    def create_deepcopy(self):
        return DataLoader(data=copy.deepcopy(self.data), dataset_name=self.dataset_name)
        
    def get_name(self):
        return self.dataset_name

    def get_number_of_sentences(self):
        return len(self.data)
    
    def get_number_of_words(self):
        return sum([len(sentence.split()) for sentence in self.data])
    
    def get_number_of_letters(self):
        #need to ensure we only count letters and not punctuation
        return sum([len(re.sub(r'[^\w\s]','',sentence)) for sentence in self.data])

    def edit_distance(reference_sentence, sentence):
        """
        Returns the number of edits required to transform reference_sentence into sentence at word level
        edits include insertions, deletions and substitutions
        based on levenshtein distance
        also returns a dictionary of substitutions, insertions and deletions
        """
        substitutions = 0
        insertions = 0
        deletions = 0
        substitution_dict = {}
        insertion_dict = {}
        deletion_dict = {}
        #remove punctuation and split into words
        sentence = re.sub(r'[^\w\s]','',sentence).lower().split()
        reference_sentence = re.sub(r'[^\w\s]','',reference_sentence).lower().split()
        #create matrix
        matrix = np.zeros((len(reference_sentence)+1,len(sentence)+1))
        #fill in first row and column
        for i in range(len(reference_sentence)+1):
            matrix[i][0] = i
        for j in range(len(sentence)+1):
            matrix[0][j] = j
        #fill in rest of matrix
        for i in range(1,len(reference_sentence)+1):
            for j in range(1,len(sentence)+1):
                if sentence[j-1] == reference_sentence[i-1]:
                    matrix[i][j] = matrix[i-1][j-1]
                else:
                    matrix[i][j] = min(matrix[i-1][j-1], matrix[i-1][j], matrix[i][j-1])+1
        #backtrack to find edits
        i = len(reference_sentence)
        j = len(sentence)
        while i > 0 and j > 0:
            if sentence[j-1] == reference_sentence[i-1]:
                i -= 1
                j -= 1
            else:
                if matrix[i][j] == matrix[i-1][j-1]+1:
                    substitutions += 1
                    if reference_sentence[i-1] not in substitution_dict:
                        substitution_dict[reference_sentence[i-1]] = {sentence[j-1]:1}
                    else:
                        if sentence[j-1] not in substitution_dict[reference_sentence[i-1]]:
                            substitution_dict[reference_sentence[i-1]][sentence[j-1]] = 1
                        else:
                            substitution_dict[reference_sentence[i-1]][sentence[j-1]] += 1
                    i -= 1
                    j -= 1
                elif matrix[i][j] == matrix[i-1][j]+1:
                    deletions += 1
                    if reference_sentence[i-1] not in deletion_dict:
                        deletion_dict[reference_sentence[i-1]] = 1
                    else:
                        deletion_dict[reference_sentence[i-1]] += 1
                    i -= 1
                elif matrix[i][j] == matrix[i][j-1]+1:
                    insertions += 1
                    if sentence[j-1] not in insertion_dict:
                        insertion_dict[sentence[j-1]] = 1
                    else:
                        insertion_dict[sentence[j-1]] += 1
                    j -= 1
        while i > 0:
            deletions += 1
            if reference_sentence[i-1] not in deletion_dict:
                deletion_dict[reference_sentence[i-1]] = 1
            else:
                deletion_dict[reference_sentence[i-1]] += 1
            i -= 1
        while j > 0:
            insertions += 1
            if sentence[j-1] not in insertion_dict:
                insertion_dict[sentence[j-1]] = 1
            else:
                insertion_dict[sentence[j-1]] += 1
            j -= 1
        distance = substitutions+insertions+deletions
        return substitutions, insertions, deletions, substitution_dict, insertion_dict, deletion_dict, distance
        
    def get_edit_distance(self, reference, manual_wer=False):
        """
        Returns the number of edits required to transform data into reference at word level, substitutions, insertions and deletions the associated dictionaries
        and the WER (withouth alignment) if manual_wer is set to True
        """
        if type(reference) == list:
            substitutions = 0
            insertions = 0
            deletions = 0
            all_sub = {}
            all_ins = {}
            all_del = {}
            distance = 0
            for i in range(len(self.data)):
                sub, ins, dele, substitution_dict, insertion_dict, deletion_dict, dist = DataLoader.edit_distance(reference[i], self.data[i], )
                all_sub = self.combine_nested_dict(all_sub, substitution_dict)
                all_ins = self.combine_dicts(all_ins, insertion_dict)
                all_del = self.combine_dicts(all_del, deletion_dict)
                substitutions += sub
                insertions += ins
                deletions += dele
                distance += dist
            if manual_wer:
                return substitutions, insertions, deletions, all_sub, all_ins, all_del, distance, distance/(sum([len(sentence.split()) for sentence in reference]))
            return substitutions, insertions, deletions, all_sub, all_ins, all_del, distance
        elif type(reference) == DataLoader:
            return self.get_edit_distance(reference.get_data(), manual_wer=manual_wer)
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")

    def get_individual_edit_distance(self, reference):
        """
        Returns the number of edits required to transform data into reference at word level for each individual sentence
        """
        if type(reference) == list:
            output = []
            for i in range(len(self.data)):
                sub, ins, dele, substitution_dict, insertion_dict, deletion_dict, distance = DataLoader.edit_distance(reference[i], self.data[i], )
                output.append((sub, ins, dele, substitution_dict, insertion_dict, deletion_dict, distance))
            return output
        elif type(reference) == DataLoader:
            return self.get_individual_edit_distance(reference.get_data())
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")       

    def combine_nested_dict(self, dict1, dict2):
        for key in dict2:
            if key not in dict1:
                dict1[key] = dict2[key]
            else:
                for key2 in dict2[key]:
                    if key2 not in dict1[key]:
                        dict1[key][key2] = dict2[key][key2]
                    else:
                        dict1[key][key2] += dict2[key][key2]
        return dict1
    
    def combine_dicts(self, dict1, dict2):
        for key in dict2:
            if key not in dict1:
                dict1[key] = dict2[key]
            else:
                dict1[key] += dict2[key]
        return dict1

    def get_bleue_score(self, reference):
        #returns bleu score of the data against a reference
        bleu = evaluate.load("bleu")
        if type(reference) == list:
            return bleu.compute(predictions=self.data, references=reference)
        elif type(reference) == DataLoader:
            return bleu.compute(predictions=self.data, references=reference.get_data())
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")

    def get_wer(self, reference):
        """
        Returns the Word Error Rate (WER) of the data against a reference. With word alignment
        """
        wer = evaluate.load("wer")
        if type(reference) == list:
            return wer.compute(predictions=self.data, references=reference)
        elif type(reference) == DataLoader:
            return wer.compute(predictions=self.data, references=reference.get_data())
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")


    def get_bert_score(self, reference):
        """
        Returns the BERT Score similarity score of the data against a reference.
        """
        bert = evaluate.load("bertscore")
        if type(reference) == list:
            return bert.compute(predictions=self.data, references=reference, lang="fr")
        elif type(reference) == DataLoader:
            return bert.compute(predictions=self.data, references=reference.get_data(), lang="fr")
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")

    def get_LaBSE(self, reference, model=None, tokenizer=None):
        """
        Returns the LaBSE similarity score of the data against a reference which is a l2 norm between the reference and target sentences score.
        Score of 1 means the sentences are identical, closer to 0 means they are less similar semantically.
        """
        if model is None:
            model = BertModel.from_pretrained("setu4993/LaBSE")
        if tokenizer is None:
            tokenizer = BertTokenizerFast.from_pretrained("setu4993/LaBSE")
        if type(reference) == list:
            pass
        elif type(reference) == DataLoader:
            reference = reference.get_data()
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")
        target = self.data
        reference_inputs = tokenizer(reference, return_tensors="pt", padding=True).to("cuda")
        target_inputs = tokenizer(target, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            reference_outputs = model(**reference_inputs)
            target_outputs = model(**target_inputs)
        reference_embeddings = reference_outputs.pooler_output
        target_embeddings = target_outputs.pooler_output
        return self.similarity(reference_embeddings, target_embeddings)
    
    def get_bleurt(self, reference, scorer = None):
        """
        BLEURT-20 is required and can be downloaded via https://github.com/google-research/bleurt
        This is the most up to date version of BLEURT and is multilingual
        Returns the BLEURT similarity score of the data against a reference.
        """
        if scorer is None:
            try:
                scorer = score.BleurtScorer("BLEURT-20")
            except:
                raise Exception("BLEURT-20 not found")
        if type(reference) == list:
            scores = scorer.score(references = reference, candidates = self.data)
        elif type(reference) == DataLoader:
            scores = scorer.score(references = reference.get_data(), candidates = self.data)
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")
        return scores
    def get_COMET(self, reference, source):
        """
        Returns the COMET similarity score of the data against a reference and a source.
        Source is the original sentence and reference is the translation
        """
        comet = evaluate.load("comet")
        if type(reference) == list:
            return comet.compute(predictions=self.data, references=reference, sources=source)
        elif type(reference) == DataLoader:
            return comet.compute(predictions=self.data, references=reference.get_data(), sources=source.get_data())
        else:
            raise Exception("Invalid reference type, please pass in a list or DataLoader instance")

In [6]:
#need to loop through file directory
aws_data = []

temp = DataLoader(path="output_data\\reddit_text\\aws\\fr.raw_reddit_text.txt", dataset_name="aws_raw_dyslexia_translated")
aws_data.append(temp)

azure_data = []

temp = DataLoader(path="output_data\\reddit_text\\azure\\raw_translated.txt", dataset_name="azure_raw_dyslexia_translated")
azure_data.append(temp)


google_data = []

temp = DataLoader(path="output_data\\reddit_text\google\\raw_reddit_text_google.docx", dataset_name="google_raw_dyslexia_translated")
google_data.append(temp)

gpt_data = []

temp = DataLoader(path="output_data\\reddit_text\gpt\gpt_raw_reddt.txt", dataset_name="gpt_raw_dyslexia_translated")
gpt_data.append(temp)



# from datasets import load_dataset
# dataset_wmt_enfr = load_dataset("wmt14",'fr-en', split='test')
to_translate_wmt14_en = []
reference_wmt14_fr = []

# for i in range(len(dataset_wmt_enfr)):
#     to_translate_wmt14_en.append(dataset_wmt_enfr[i]['translation']['en'])
#     reference_wmt14_fr.append(dataset_wmt_enfr[i]['translation']['fr'])

aws_reference_corpus_fr = DataLoader(path="output_data\\reddit_text\\aws\\fr.corrected_reddit_text.txt", dataset_name="aws_reddit_reference")
azure_reference_corpus_fr = DataLoader(path="output_data\\reddit_text\\azure\corrected_translated.txt", dataset_name="azure_reddit_reference")
google_reference_corpus_fr = DataLoader(path="output_data\\reddit_text\google\corrected_reddit_text_google.docx", dataset_name="google_reddit_reference")
gpt_reference_corpus_fr = DataLoader(path="output_data\\reddit_text\gpt\gpt_corrected_reddt.txt", dataset_name="gpt_reddit_reference")
reference_corpus_en = DataLoader(path="r_Dyslexia_Text\default_files\corrected\corrected_reddit_text.txt", dataset_name="english_reddit_corrected")


In [7]:
aws_bleu_scores = []
for data in aws_data:
    print(data.get_name())
    aws_bleu_scores.append(data.get_bleue_score(aws_reference_corpus_fr))
print(f"AWS BLEU scores: {aws_bleu_scores}")

google_bleu_scores = []
for data in google_data:
    print(data.get_name())
    google_bleu_scores.append(data.get_bleue_score(google_reference_corpus_fr))
print(f"Google BLEU scores: {google_bleu_scores}")

azure_bleu_scores = []
for data in azure_data:
    print(data.get_name())
    azure_bleu_scores.append(data.get_bleue_score(azure_reference_corpus_fr))
print(f"Azure BLEU scores: {azure_bleu_scores}")

gpt_bleu_scores = []
for data in gpt_data:
    print(data.get_name())
    gpt_bleu_scores.append(data.get_bleue_score(gpt_reference_corpus_fr))
print(f"GPT BLEU scores: {gpt_bleu_scores}")

#same of above but for WER
aws_wer_scores = []
for data in aws_data:
    print(data.get_name())
    aws_wer_scores.append(data.get_wer(aws_reference_corpus_fr))
print(f"AWS WER scores: {aws_wer_scores}")

google_wer_scores = []
for data in google_data:
    print(data.get_name())
    google_wer_scores.append(data.get_wer(google_reference_corpus_fr))
print(f"Google WER scores: {google_wer_scores}")

azure_wer_scores = []
for data in azure_data:
    print(data.get_name())
    azure_wer_scores.append(data.get_wer(azure_reference_corpus_fr))
print(f"Azure WER scores: {azure_wer_scores}")

gpt_wer_scores = []
for data in gpt_data:
    print(data.get_name())
    gpt_wer_scores.append(data.get_wer(gpt_reference_corpus_fr))
print(f"GPT WER scores: {gpt_wer_scores}")


aws_raw_dyslexia_translated
AWS BLEU scores: [{'bleu': 0.8154814184130349, 'precisions': [0.9165835825855766, 0.8456398262612763, 0.7940880080618072, 0.7487335359675785], 'brevity_penalty': 0.9897504621546946, 'length_ratio': 0.9898026315789473, 'translation_length': 3009, 'reference_length': 3040}]
google_raw_dyslexia_translated
Google BLEU scores: [{'bleu': 0.8583347905387105, 'precisions': [0.936450444517616, 0.879179079774909, 0.8369384359400999, 0.8002676480428237], 'brevity_penalty': 0.9960565282884352, 'length_ratio': 0.9960642833715972, 'translation_length': 3037, 'reference_length': 3049}]
azure_raw_dyslexia_translated
Azure BLEU scores: [{'bleu': 0.8355527156326809, 'precisions': [0.926829268292683, 0.8642929123278468, 0.8166160081053698, 0.7735144312393888], 'brevity_penalty': 0.9906884613430641, 'length_ratio': 0.9907315458457464, 'translation_length': 2993, 'reference_length': 3021}]
gpt_raw_dyslexia_translated
GPT BLEU scores: [{'bleu': 0.6420393638140868, 'precisions': [

In [8]:
# aws_COMET_scores = []
# for data in aws_data:
#     print(data.get_name())
#     aws_COMET_scores.append(data.get_COMET(aws_reference_corpus_fr, reference_corpus_en))
# print(f"AWS COMET scores: {aws_COMET_scores}")


# google_COMET_scores = []
# for data in google_data:
#     print(data.get_name())
#     google_COMET_scores.append(data.get_COMET(google_reference_corpus_fr, reference_corpus_en))
# print(f"Google COMET scores: {google_COMET_scores}")


# azure_COMET_scores = []
# for data in azure_data:
#     print(data.get_name())
#     azure_COMET_scores.append(data.get_COMET(azure_reference_corpus_fr, reference_corpus_en))
# print(f"Azure COMET scores: {azure_COMET_scores}")

# gpt_COMET_scores = []
# for data in gpt_data:
#     print(data.get_name())
#     gpt_COMET_scores.append(data.get_COMET(gpt_reference_corpus_fr, reference_corpus_en))
# print(f"GPT COMET scores: {gpt_COMET_scores}")

# #load COMET scores from pkl file

# # aws_COMET_scores_v1 = pickle.load(open("COMET_scores/aws_COMET_scores_v1.pkl", "rb"))
# # aws_COMET_scores_v2 = pickle.load(open("COMET_scores/aws_COMET_scores_v2.pkl", "rb"))
# # google_COMET_scores_v1 = pickle.load(open("COMET_scores/google_COMET_scores_v1.pkl", "rb"))
# # google_COMET_scores_v2 = pickle.load(open("COMET_scores/google_COMET_scores_v2.pkl", "rb"))
# # azure_COMET_scores_v1 = pickle.load(open("COMET_scores/azure_COMET_scores_v1.pkl", "rb"))
# # azure_COMET_scores_v2 = pickle.load(open("COMET_scores/azure_COMET_scores_v2.pkl", "rb"))
# # gpt_COMET_scores_v1 = pickle.load(open("COMET_scores/gpt_COMET_scores_v1.pkl", "rb"))
# # gpt_COMET_scores_v2 = pickle.load(open("COMET_scores/gpt_COMET_scores_v2.pkl", "rb"))


In [9]:
# scorer = score.BleurtScorer("BLEURT-20")
# aws_BLEURT_scores = []
# for data in aws_data:
#     print(data.get_name())
#     aws_BLEURT_scores.append(data.get_bleurt(aws_reference_corpus_fr, scorer = scorer))
# print(f"AWS BLEURT scores: {aws_BLEURT_scores}")


# google_BLEURT_scores = []
# for data in google_data:
#     print(data.get_name())
#     google_BLEURT_scores.append(data.get_bleurt(google_reference_corpus_fr, scorer = scorer))
# print(f"Google BLEURT scores: {google_BLEURT_scores}")


# azure_BLEURT_scores = []
# for data in azure_data:
#     print(data.get_name())
#     azure_BLEURT_scores.append(data.get_bleurt(azure_reference_corpus_fr, scorer = scorer))
# print(f"Azure BLEURT scores: {azure_BLEURT_scores}")

# gpt_BLEURT_scores = []
# for data in gpt_data:
#     print(data.get_name())
#     gpt_BLEURT_scores.append(data.get_bleurt(gpt_reference_corpus_fr, scorer = scorer))
# print(f"GPT BLEURT scores: {gpt_BLEURT_scores}")

aws_BLEURT_scores = pickle.load(open("BLEURT_scores/reddit_aws_bleuRT_scores.pkl", "rb"))

google_BLEURT_scores = pickle.load(open("BLEURT_scores/reddit_google_bleuRT_scores.pkl", "rb"))

azure_BLEURT_scores = pickle.load(open("BLEURT_scores/reddit_azure_bleuRT_scores.pkl", "rb"))

gpt_BLEURT_scores = pickle.load(open("BLEURT_scores/reddit_gpt_bleuRT_scores.pkl", "rb"))



In [10]:
# #save BLEURT scores to pickle files
# import pickle
# with open("BLEURT_scores/reddit_aws_bleuRT_scores.pkl", "wb") as f:
#     pickle.dump(aws_BLEURT_scores, f)

# with open("BLEURT_scores/reddit_google_bleuRT_scores.pkl", "wb") as f:
#     pickle.dump(google_BLEURT_scores, f)

# with open("BLEURT_scores/reddit_azure_bleuRT_scores.pkl", "wb") as f:
#     pickle.dump(azure_BLEURT_scores, f)

# with open("BLEURT_scores/reddit_gpt_bleuRT_scores.pkl", "wb") as f:
#     pickle.dump(gpt_BLEURT_scores, f)


In [11]:
aws_bert_scores = []
for data in aws_data:
    print(data.get_name())
    aws_bert_scores.append(data.get_bert_score(aws_reference_corpus_fr))
print(f"AWS bert scores: {aws_bert_scores}")


google_bert_scores = []
for data in google_data:
    print(data.get_name())
    google_bert_scores.append(data.get_bert_score(google_reference_corpus_fr))
print(f"Google bert scores: {google_bert_scores}")


azure_bert_scores = []
for data in azure_data:
    print(data.get_name())
    azure_bert_scores.append(data.get_bert_score(azure_reference_corpus_fr))
print(f"Azure bert scores: {azure_bert_scores}")

gpt_bert_scores = []
for data in gpt_data:
    print(data.get_name())
    gpt_bert_scores.append(data.get_bert_score(gpt_reference_corpus_fr))
print(f"GPT bert scores: {gpt_bert_scores}")

aws_raw_dyslexia_translated


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

c:\Users\User\anaconda3\envs\dyslexia\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--bert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

AWS bert scores: [{'precision': [0.9479723572731018, 0.9794503450393677, 0.9612154960632324, 0.8883624076843262, 0.9742841720581055, 0.991306722164154, 0.9655714631080627, 0.9402908086776733, 0.9099382758140564, 0.8624309301376343, 0.913024365901947, 0.9785767197608948, 0.9777623414993286, 0.991546630859375, 0.9381820559501648, 0.939733624458313], 'recall': [0.941684365272522, 0.97940593957901, 0.9608500003814697, 0.878879725933075, 0.9759867191314697, 0.9926636219024658, 0.9631620645523071, 0.9285342693328857, 0.9226660132408142, 0.8539584875106812, 0.9117687940597534, 0.9768517017364502, 0.9791690707206726, 0.9955906867980957, 0.9293309450149536, 0.9325370788574219], 'f1': [0.944817841053009, 0.9794281125068665, 0.9610327482223511, 0.8835955858230591, 0.9751346707344055, 0.9919846653938293, 0.9643652439117432, 0.9343755841255188, 0.916257917881012, 0.8581738471984863, 0.9123961925506592, 0.9777134656906128, 0.9784652590751648, 0.9935645461082458, 0.9337355494499207, 0.936121523380279